# Experimental transfer and online fine-tuning

Selected from the original unsuffixed notebook: its saved outputs at zero-based cell 33 match the chapter learning/reward figures byte for byte. The exact weight snapshot is not established by that match.

Selected source excerpts from the original research notebook. Outputs, execution counts, widget state and duplicate plotting trials are removed. Definitions are kept as code; configuration, training launches and hardware calls are displayed as reference text so Run All cannot start training or connect to a robot. This is a review extract, not a complete reproducibility package. See README.md for omitted dependencies.


In [ ]:
import os
import random
from collections import deque
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
device = torch.device("cpu")


## Source cell 3 (original zero-based index 2)


Reference only; not executed.

```python
import os
import torch
import random
import numpy as np
import torch.nn as nn
from tqdm import tqdm
import torch.optim as optim
from scipy.io import loadmat
from collections import deque

from utils import RobotCMD
from utils import Cameras
from utils import ExperimentalEnv

# Configuration
current_dir = os.getcwd()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Path configurations
EXP_NAME = '500 eps-Rand-cube-10 HZ'
PATHS = {
    # Weights
    'best_weights': os.path.join(current_dir, 'Weights', EXP_NAME, 'Best_weights'),
    'last_weights': os.path.join(current_dir, 'Weights', EXP_NAME, 'Last_weights'),
    'fine_tuned_weights': os.path.join(current_dir, 'Weights', EXP_NAME, 'FineTuned_weights'),

    # Variables
    'variables': os.path.join(current_dir, 'Variables', EXP_NAME),

    # Renders
    'renders_train': os.path.join(current_dir, 'Renders', EXP_NAME, 'Train_DRL'),
    'render_tests': os.path.join(current_dir, 'Renders', EXP_NAME, 'Test_DRL'),

    # Plots
    'plots_generated_points': os.path.join(current_dir, 'Plots', EXP_NAME, 'Generated_Points'),
    'plots_learning': os.path.join(current_dir, 'Plots', EXP_NAME, 'Learning_Curves'),
    'plots_test': os.path.join(current_dir, 'Plots', EXP_NAME, 'Tests'),
    'plots_train': os.path.join(current_dir, 'Plots', EXP_NAME, 'Trains'),

    # Hardware Specific
    'camera': os.path.join(current_dir, 'Cameras'),

    # MATLAB workspaces
    'matlab_workspaces': os.path.abspath(os.path.join(current_dir, '2) Workspaces', 'Cartesian'))
}

ROBOT_IP = "ROBOT_HOST_FROM_LOCAL_CONFIGURATION"
ROBOT_PORT = 8055
CONT_DT = 0.1  # 10 Hz

for path in PATHS.values():
    os.makedirs(path, exist_ok=True)
```


## Source cell 8 (original zero-based index 7)


In [ ]:
class Actor(nn.Module):
    def __init__(self, actor_input_dim, action_dim, action_max, action_min, actor_hidden, input_min, input_max):
        super().__init__()
        self.register_buffer('action_max', torch.tensor(action_max, dtype=torch.float32))
        self.register_buffer('action_min', torch.tensor(action_min, dtype=torch.float32))
        self.register_buffer('input_min', torch.tensor(input_min, dtype=torch.float32))
        self.register_buffer('input_max', torch.tensor(input_max, dtype=torch.float32))
        layers = []
        for size in actor_hidden:
            layers.extend([nn.Linear(actor_input_dim, size), nn.LayerNorm(size), nn.ReLU()])
            actor_input_dim = size
        layers.append(nn.Linear(actor_input_dim, action_dim))
        self.net = nn.Sequential(*layers).to(device)
        self.apply(self._init_weights)
   
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
            nn.init.zeros_(m.bias)
   
    def forward(self, x):
        x_norm = 2 * (x - self.input_min) / (self.input_max - self.input_min) - 1
        act_norm = torch.tanh(self.net(x_norm))
        act = 0.5 * (act_norm + 1) * (self.action_max - self.action_min) + self.action_min
        return act


## Source cell 10 (original zero-based index 9)


In [ ]:
class Critic(nn.Module):
    def __init__(self, critic_input_dim, hidden_sizes, input_min, input_max):
        super().__init__()
        self.register_buffer('input_min', torch.tensor(input_min, dtype=torch.float32))
        self.register_buffer('input_max', torch.tensor(input_max, dtype=torch.float32))
        layers = []
        for size in hidden_sizes:
            layers.extend([nn.Linear(critic_input_dim, size), nn.LayerNorm(size), nn.ReLU()])
            critic_input_dim = size
        layers.append(nn.Linear(critic_input_dim, 1))
        self.q_net = nn.Sequential(*layers).to(device)
        self.apply(self._init_weights)
   
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
            nn.init.zeros_(m.bias)
   
    def forward(self, x):
        x_norm = 2 * (x - self.input_min) / (self.input_max - self.input_min) - 1
        return self.q_net(x_norm)


## Source cell 12 (original zero-based index 11)


In [ ]:
class ReplayBuffer:
    def __init__(self, state_dim, action_dim, capacity=int(1e6)):
        self.capacity = capacity
        self.idx = 0
        self.full = False
        
        self.states = torch.zeros((capacity, state_dim), device=device, dtype=torch.float32)
        self.actions = torch.zeros((capacity, action_dim), device=device, dtype=torch.float32)
        self.rewards = torch.zeros(capacity, device=device, dtype=torch.float32)

    def add(self, state, action, reward):
        idx = self.idx
        
        self.states[idx] = torch.as_tensor(state, device=device, dtype=torch.float32)
        self.actions[idx] = torch.as_tensor(action, device=device, dtype=torch.float32)
        self.rewards[idx] = torch.as_tensor(reward, device=device, dtype=torch.float32)
        
        self.idx = (self.idx + 1) % self.capacity
        self.full = self.full or (self.idx == 0)

    def sample(self, batch_size):
        max_idx = self.capacity if self.full else self.idx
        indices = torch.randint(0, max_idx, (batch_size,), device=device)
        
        return (
            self.states[indices],
            self.actions[indices],
            self.rewards[indices],
        )


## Source cell 21 (original zero-based index 20)


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pickle # Added for buffer save/load
from tqdm import tqdm
from scipy.io import loadmat

class DDPGFineTuner:
    def __init__(self, env, actor_hidden=(400, 300), critic_hidden=(400, 300),
                 lr_actor=1e-5, lr_critic=5e-5, batch_size=64,
                 noise_init=0.015, noise_decay=0.995, noise_min=0.005):

        self.env = env
        self.batch_size = batch_size
        self.plot_dir = PATHS['plots_train']
        self.plots_points_dir = PATHS['plots_generated_points']
        
        self.all_ep_tips = []
        self.noise_scale, self.noise_decay, self.noise_min = noise_init, noise_decay, noise_min

        self.state_dim, self.action_dim = env.observation_space.shape[0], env.action_space.shape[0]           
        
        act_low, act_high = env.action_space.low, env.action_space.high
        state_low, state_high = env.observation_space.low, env.observation_space.high
        
        self.action_low_t = torch.tensor(act_low, dtype=torch.float32, device=device)
        self.action_high_t = torch.tensor(act_high, dtype=torch.float32, device=device)
        self.action_range = torch.tensor((act_high - act_low) / 6.0, device=device) 

        # --- Networks ---
        self.actor = Actor(self.state_dim, self.action_dim, act_high, act_low, actor_hidden, state_low, state_high).to(device)
        self.critic = Critic(self.state_dim + self.action_dim, critic_hidden, 
                             np.concatenate([state_low, act_low]), np.concatenate([state_high, act_high])).to(device)

        # Strictly load pre-trained weights to prevent erratic hardware behavior
        try:
            self.actor.load_state_dict(torch.load(os.path.join(PATHS['best_weights'], 'actor_dict.pth'), map_location=device))
            self.critic.load_state_dict(torch.load(os.path.join(PATHS['best_weights'], 'critic_dict.pth'), map_location=device))
            print("[SUCCESS] Loaded pre-trained simulation weights for fine-tuning.")
        except FileNotFoundError:
            raise RuntimeError("CRITICAL: Simulation weights not found. Fine-tuning from scratch will damage hardware.")

        self.actor_optim = optim.Adam(self.actor.parameters(), lr=lr_actor)
        self.critic_optim = optim.Adam(self.critic.parameters(), lr=lr_critic)
        
        # --- Replay Buffer Initialization & Loading ---
        self.buffer = ReplayBuffer(self.state_dim, self.action_dim)
        self.buffer_path = os.path.join(PATHS['fine_tuned_weights'], 'replay_buffer.pkl')
        
        if os.path.exists(self.buffer_path):
            try:
                with open(self.buffer_path, 'rb') as f:
                    self.buffer = pickle.load(f)
                print(f"[SUCCESS] Loaded existing replay buffer from {self.buffer_path} | Current Size: {self.buffer.idx}")
            except Exception as e:
                print(f"[WARNING] Could not load replay buffer, starting fresh. Error: {e}")
        else:
            print("[INFO] No existing replay buffer found. Starting fresh.")
        
        self.history = {k: [] for k in ['returns', 'success_rates', 'step_term', 
                                        'critic_losses', 'actor_losses', 'pos_norm', 'v_norm', 'a_norm',
                                        'r_pos', 'r_v']}

    def store_history(self, ep_reward, ep_success, s_term, cl_loss, al_loss, metrics):
        h = self.history
        h['returns'].append(ep_reward)
        h['success_rates'].append(ep_success)
        h['step_term'].append(s_term)
        h['critic_losses'].append(np.mean(cl_loss) if cl_loss else 0.0)
        h['actor_losses'].append(np.mean(al_loss) if al_loss else 0.0)
        for k in ['pos_norm', 'v_norm', 'a_norm', 'r_pos', 'r_v']:
            h[k].append(np.mean(metrics[k]) if metrics[k] else 0.0)

    def act(self, state, training=True):
        with torch.no_grad():
            state_t = torch.as_tensor(state, dtype=torch.float32, device=device)
            action = self.actor(state_t)
            if training:
                action += torch.normal(0.0, self.action_range * self.noise_scale)
            return torch.clamp(action, self.action_low_t, self.action_high_t).cpu().numpy()

    def update(self):
        states, actions, rewards = self.buffer.sample(self.batch_size)
        rewards = rewards.unsqueeze(1)

        # --- Critic Update ---
        with torch.no_grad():
            target_q = rewards

        current_q = self.critic(torch.cat([states, actions], dim=1))
        critic_loss = nn.MSELoss()(current_q, target_q)
        
        self.critic_optim.zero_grad()
        critic_loss.backward()
        self.critic_optim.step()
        
        # --- Actor Update ---
        for p in self.critic.parameters(): p.requires_grad = False
        actor_loss = -self.critic(torch.cat([states, self.actor(states)], dim=1)).mean()
        self.actor_optim.zero_grad()
        actor_loss.backward()
        self.actor_optim.step()
        for p in self.critic.parameters(): p.requires_grad = True
            
        return critic_loss.item(), actor_loss.item()
    
    def train(self, selected_datas, episodes=20, steps=100):
        best_reward = float('-inf')

        # datas = loadmat(DATASETS['train'][0])
        # pos_setpoints = datas['datas']['Pos'][0, 0][:, 0:3]   

        for ep in range(1, episodes + 1):
            #### using Single Trajectory###################################################
            # datas = loadmat(DATASETS['train'][0])
            # pos_setpoints = datas['datas']['Pos'][0, 0][:, 0:3]
            # action0 = [0.0, 0.0, 0.20]
            ###############################################################################

            #### using Two Trajectories####################################################
            # Put the two dataset options into a list
            dataset_options = [DATASETS['train'][0], DATASETS['test'][1]]
            # Choose randomly between the two
            selected_dataset_path = random.choice(dataset_options)
            # Load the randomly selected dataset
            datas = loadmat(selected_dataset_path)
            pos_setpoints = datas['datas']['Pos'][0, 0][:, 0:3]
            action0 = [0.0, 0.0, 0.20]
            ###############################################################################

            #### using the selected datpoints##############################################
            # # 1. Randomly pick one of the representative episode IDs
            # chosen_ep_id = np.random.choice(selected_datas)
            
            # # 2. Extract the corresponding trajectory (X, Y, Z positions)
            # ep_data = self.env.cleaned_dataset[self.env.cleaned_dataset[:, 0] == chosen_ep_id]
            # pos_setpoints = ep_data[:, 2:5]

            # p_tip_t_local, v_tip_t_local = self.env.camera.get_state()
            # wait_attempts = 0
            # while np.all(p_tip_t_local == 0.0) and wait_attempts < 50:
            #     time.sleep(0.1)
            #     p_tip_t_local, v_tip_t_local = self.env.camera.get_state()
            #     wait_attempts += 1
                
            # if np.all(p_tip_t_local == 0.0):
            #     print("[WARNING] Reset tip detection failed. Camera returned zeros.")

            # p_tip_t = p_tip_t_local
            # v_tip_t = v_tip_t_local

            # p_target = pos_setpoints[0]
            # self.env.camera.update_setpoint(p_target)
            # error_tip_t = p_tip_t - p_target
            # state = np.concatenate([p_tip_t, error_tip_t, v_tip_t], dtype=np.float32)
            # action0 = self.act(state, training=False)
            ##############################################################################

            # action0 = None
            state = self.env.reset(action0, render=True)

            ep_reward, ep_success, step_term = 0, 0, 0
            cl_loss, al_loss, ep_p_tips = [], [], [] 
            
            # Match dictionary keys exactly to the environment's returned info keys
            metrics_keys = ['pos_norm', 'v_norm', 'a_norm', 'r_pos', 'r_v']
            metrics = {k: [] for k in metrics_keys}
            run = {k: 0.0 for k in metrics_keys}
            run.update({'cl': 0.0, 'al': 0.0})
            upd_cnt = 0
                
            with tqdm(total=steps, desc=f'Ep {ep}/{episodes}', unit='step') as pbar:
                for step in range(steps):

                    self.env.p_target = pos_setpoints[step]
                    self.env.camera.update_setpoint(self.env.p_target)
                    self.env.error_tip_t = self.env.p_tip_t - self.env.p_target
                    state = self.env._get_state() 

                    action = self.act(state, training=True)
                    next_state, reward, info = self.env.step(action)
                    
                    if info.get('step_term', False):
                        state = self.env.reset(action0, reder=True)
                        step_term += 1
                    else:
                        self.buffer.add(state, action, reward)
                        ep_p_tips.append(self.env.p_target) 
                        ep_success += int(info.get('is_success', False))
                            
                        for k in metrics_keys: 
                            val = info.get(k, 0.0)
                            metrics[k].append(val)
                            run[k] += val
                        
                        state = next_state
                        ep_reward += reward

                        if self.buffer.idx > self.batch_size:
                            cl, al = self.update()
                            cl_loss.append(cl)
                            al_loss.append(al)
                            run['cl'] += cl
                            run['al'] += al
                            upd_cnt += 1

                            s_div = step + 1
                            pbar.set_postfix_str(
                                f"CL: {run['cl']/upd_cnt:.3f} | AL: {run['al']/upd_cnt:.3f} | Rew: {ep_reward:.1f} | Succ: {(ep_success/steps)*100:.1f} | "
                                f"PE: {(run['pos_norm']/s_div)*1000:.1f} | RP: {run['r_pos']/s_div:.2f} | "
                                f"VE: {(run['v_norm']/s_div)*1000:.1f} | RV: {run['r_v']/s_div:.2f} | "
                                f"AE: {(run['a_norm']/s_div)*1000:.1f} | S_T: {step_term}"
                            )

                    pbar.update(1)

            self.all_ep_tips.extend(ep_p_tips)
            self.noise_scale = max(self.noise_scale * self.noise_decay, self.noise_min)
            self.store_history(ep_reward, (ep_success/steps)*100, step_term, cl_loss, al_loss, metrics)

            # Save to fine_tuned directories
            torch.save(self.actor.state_dict(), os.path.join(PATHS['fine_tuned_weights'], 'actor_dict_last.pth'))
            torch.save(self.critic.state_dict(), os.path.join(PATHS['fine_tuned_weights'], 'critic_dict_last.pth'))
            
            if ep_reward >= best_reward:
                best_reward = ep_reward
                torch.save(self.actor.state_dict(), os.path.join(PATHS['fine_tuned_weights'], 'actor_dict_best.pth'))
                torch.save(self.critic.state_dict(), os.path.join(PATHS['fine_tuned_weights'], 'critic_dict_best.pth'))
                
            # --- Save Buffer Every 25 Episodes ---
            if ep % 25 == 0:
                with open(self.buffer_path, 'wb') as f:
                    pickle.dump(self.buffer, f)
                print(f"[INFO] Replay buffer saved at episode {ep} to {self.buffer_path}")
                
            # Save visuals
            if ep % 25 == 0 or ep in [10]:
                save_path = os.path.join(PATHS['plots_generated_points'], f'random_points_up_to_ep_{ep}.png')
                plot_3d_random_points(self.all_ep_tips, save_path, f'Cumulative Target Points up to Episode {ep}')

            if ep % 25 == 0 or ep in [10]:
                print(f"Rendering evaluation for episode {ep}...")
                render_file = DATASETS['train'][0]
                plot_path_pos = os.path.join(self.plot_dir, f'episode_{ep}_pos_time.png')
                plot_path_traj = os.path.join(self.plot_dir, f'episode_{ep}_Trajectory.png')                
                video_path = os.path.join(self.plot_dir, f'episode_{ep}_camera_tracking.avi')
                try:
                    # Start recording the OpenCV window
                    if hasattr(self.env, 'camera'):
                        self.env.camera.start_recording(video_path)
                    render_movement(self, self.env, render_file, plot_path_pos, plot_path_traj, show=False)
                    
                except AttributeError as e:
                    print(f"Skipping visualization creation: {e}")
                finally:
                    # Ensure the video file is closed and saved properly
                    if hasattr(self.env, 'camera'):
                        self.env.camera.stop_recording()

        return self.history, self.actor, self.critic


## Source cell 23 (original zero-based index 22)


Reference only; not executed.

```python
# 1. Initialize Robot via the new RobotCMD wrapper
robot = RobotCMD(ip=ROBOT_IP, port=ROBOT_PORT)
if not robot.connect():
    raise RuntimeError("Robot connection failed.")
```


## Source cell 24 (original zero-based index 23)


Reference only; not executed.

```python
# 2. Initialize Dual Cameras
eul_t = np.array([np.deg2rad(180.0), np.deg2rad(-90), np.deg2rad(0)])
camera = Cameras(eul_t=eul_t, cameras_folder=PATHS['camera'])
if not camera.initialize_cameras():
    raise RuntimeError("Camera connection failed.")
camera.start_visualization()
print("Cameras successfully initialized and visualization thread started.")
```


## Source cell 25 (original zero-based index 24)


Reference only; not executed.

```python
# 3. Bind Hardware to Gym Environment
dataset_path = os.path.join(current_dir, 'Catheter_DL_Dataset.mat')
env = ExperimentalEnv(
    L0=0.1050, 
    T0=0.0033, 
    mag_l=0.09, 
    dataset_path=dataset_path, 
    camera=camera, 
    robot=robot, 
    cont_dt=CONT_DT
)
```


## Source cell 28 (original zero-based index 27)


Reference only; not executed.

```python
%%time

# agent = DDPGFineTuner(
#     env=env,
#     actor_hidden=(400, 300),
#     critic_hidden=(400, 300),
#     lr_actor=1e-5,
#     lr_critic=5e-5,
#     batch_size=64,       
#     noise_init=0.05,
#     noise_decay=0.9,
#     noise_min=0.01
# )

agent = DDPGFineTuner(
    env=env,
    actor_hidden=(400, 300),
    critic_hidden=(400, 300),
    lr_actor=5e-5,
    lr_critic=1e-4,
    # lr_actor=1e-4,
    # lr_critic=5e-4,
    batch_size=256,
    noise_init=0.05,
    noise_decay=0.9,
    noise_min=0.01
)

# The train function now returns more metrics
# (history, actor_model, critic_model) = agent.train(episodes=10, steps=500)
(history, actor_model, critic_model) = agent.train(episodes=200, steps=500, selected_datas=selected_datas)

env.close()
```
